In [2]:
import warnings
warnings.filterwarnings("ignore")
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

In [ ]:
model_name = "mosaicml/mpt-7b"
local_dir = os.path.join("pre-trained")
cache_dir = os.path.join("hf-cache")

# Check if model is already saved locally
if not os.path.exists(local_dir) or not os.path.exists(os.path.join(local_dir, "config.json")):
    print("Model not found locally. Downloading and saving...")
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        cache_dir=cache_dir
    )
    model.save_pretrained(local_dir)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        cache_dir=cache_dir
    )
    tokenizer.save_pretrained(local_dir)
else:
    print("Model found locally. Loading from disk...")

# Load model and tokenizer from local_dir
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    local_dir,
    trust_remote_code=True
)

Model not found locally. Downloading and saving...


In [ ]:

# Load your long descriptions
with open('texts_cleaned.json', 'r') as f:
    image_to_text_mapping = json.load(f)

text_embeddings_dict = {}

for image_name, text in image_to_text_mapping.items():
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
            hidden = outputs.hidden_states[-1]  # Last layer
            pooled = hidden.mean(dim=1).squeeze().cpu().numpy()
            text_embeddings_dict[image_name] = pooled
            print(f"Embedded: {image_name}")
    except Exception as e:
        print(f"Error: {image_name} - {e}")

# np.save("text_embeddings_mpt7b.npy", text_embeddings_dict)
print(f"Saved {len(text_embeddings_dict)} text embeddings")

In [3]:
class TextEmbed:
    def __init__(self):
        model_name = "mosaicml/mpt-7b"
        local_dir = os.path.join("pre-trained")
        cache_dir = os.path.join("hf-cache")
        
        # Check if model is already saved locally
        if not os.path.exists(local_dir) or not os.path.exists(os.path.join(local_dir, "config.json")):
            print("Model not found locally. Downloading and saving...")
            
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                torch_dtype=torch.float16,
                cache_dir=cache_dir
            )
            self.model.save_pretrained(local_dir)
        
            self.tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True,
                cache_dir=cache_dir
            )
            tokenizer.save_pretrained(local_dir)
        else:
            print("Model found locally. Loading from disk...")
        
        # Load model and tokenizer from local_dir
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.model = AutoModelForCausalLM.from_pretrained(
            local_dir,
            trust_remote_code=True,
            torch_dtype=torch.float16
        ).to(device).eval()
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            local_dir,
            trust_remote_code=True
        )
        self.device = device

text_embedder = TextEmbed()

text_encoder, tokenizer = text_embedder.model, text_embedder.tokenizer

Model not found locally. Downloading and saving...


KeyboardInterrupt: 